# Student Exam Score Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `exam_score`

## 2 · Project Overview

We predict **student exam scores** (0-100) based on study habits (hours, attendance), academic history (GPA), demographics (parent education, internet access), and school characteristics (type, class size, tutoring).

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a student's study hours, attendance, previous GPA, sleep, parental education, tutoring, school type, extracurriculars, class size, and internet access, predict their exam score.

## 5 · Why This Project Matters

- **Educational outcome prediction** helps identify at-risk students early.
- Understanding which factors drive scores enables targeted interventions.
- Resource allocation (tutoring, class size) can be data-driven.
- Demonstrates regression with mixed continuous and categorical features.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,000 |
| **Features** | study_hours_per_week, attendance_pct, previous_gpa, sleep_hours, parent_education, tutoring, school_type, extracurricular_hours, class_size, has_internet |
| **Target** | exam_score (continuous, 0-100) |
| **Range** | ~10 – 100 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "exam_score"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 7,000 student records with study habits, demographics, and school features.

In [ ]:
np.random.seed(SEED)
n = 7000
study_hours_per_week = np.round(np.random.lognormal(2.0, 0.5, n).clip(0.5, 50), 1)
attendance_pct = np.round(np.random.beta(6, 2, n) * 100, 1).clip(20, 100)
previous_gpa = np.round(np.random.normal(3.0, 0.5, n).clip(1.0, 4.0), 2)
sleep_hours = np.round(np.random.normal(7, 1.2, n).clip(3, 11), 1)
parent_education = np.random.choice(["High School", "Bachelor", "Master", "PhD"], n,
                                     p=[0.3, 0.35, 0.25, 0.1])
parent_ed_val = {"High School": 0, "Bachelor": 3, "Master": 5, "PhD": 7}
ped_val = np.array([parent_ed_val[p] for p in parent_education])

tutoring = np.random.choice(["None", "Occasional", "Regular"], n, p=[0.5, 0.3, 0.2])
tutor_val = {"None": 0, "Occasional": 3, "Regular": 6}
tut_val = np.array([tutor_val[t] for t in tutoring])

school_type = np.random.choice(["Public", "Private", "Charter"], n, p=[0.55, 0.25, 0.2])
school_mult = {"Public": 1.0, "Private": 1.05, "Charter": 1.02}
sch_val = np.array([school_mult[s] for s in school_type])

extracurricular_hours = np.round(np.random.exponential(3, n).clip(0, 25), 1)
class_size = np.random.randint(15, 45, n)
has_internet = np.random.binomial(1, 0.85, n)

# Exam score model (0-100)
exam_score = (25
    + 1.2 * study_hours_per_week
    + 0.15 * attendance_pct
    + 6 * previous_gpa
    + 0.8 * sleep_hours
    + ped_val
    + tut_val
    - 0.1 * class_size
    + 2 * has_internet
    - 0.2 * np.maximum(extracurricular_hours - 10, 0))
exam_score = exam_score * sch_val
exam_score = np.round(exam_score + np.random.normal(0, 5, n), 1).clip(10, 100)

df = pd.DataFrame({
    "study_hours_per_week": study_hours_per_week, "attendance_pct": attendance_pct,
    "previous_gpa": previous_gpa, "sleep_hours": sleep_hours,
    "parent_education": parent_education, "tutoring": tutoring,
    "school_type": school_type, "extracurricular_hours": extracurricular_hours,
    "class_size": class_size, "has_internet": has_internet,
    "exam_score": exam_score
})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['exam_score'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["study_hours_per_week"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Study Hours/Week"); axes[0][0].set_ylabel("Exam Score")
axes[0][0].set_title("Study Hours vs Score")

axes[0][1].scatter(df["attendance_pct"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Attendance (%)"); axes[0][1].set_ylabel("Exam Score")
axes[0][1].set_title("Attendance vs Score")

axes[0][2].scatter(df["previous_gpa"], df[TARGET], alpha=0.2, s=10)
axes[0][2].set_xlabel("Previous GPA"); axes[0][2].set_ylabel("Exam Score")
axes[0][2].set_title("Previous GPA vs Score")

df.boxplot(column=TARGET, by="parent_education", ax=axes[1][0])
axes[1][0].set_title("Score by Parent Education")
axes[1][0].tick_params(axis="x", rotation=45)

df.boxplot(column=TARGET, by="tutoring", ax=axes[1][1])
axes[1][1].set_title("Score by Tutoring")

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink": 0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `exam_score`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Exam Score")

df.boxplot(column=TARGET, by="tutoring", ax=axes[1])
axes[1].set_title("Exam Score by Tutoring Level")

plt.tight_layout()
plt.show()
print(f"Range: {df[TARGET].min():.1f} to {df[TARGET].max():.1f}")
print(f"Mean: {df[TARGET].mean():.1f}, Median: {df[TARGET].median():.1f}")
print(f"Std: {df[TARGET].std():.1f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `parent_education`, `tutoring`, `school_type` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: Extreme low scores may indicate data quality issues — keep for robustness.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["study_efficiency"] = X_train["study_hours_per_week"] * X_train["attendance_pct"] / 100
X_test["study_efficiency"] = X_test["study_hours_per_week"] * X_test["attendance_pct"] / 100

X_train["sleep_deficit"] = np.maximum(8 - X_train["sleep_hours"], 0)
X_test["sleep_deficit"] = np.maximum(8 - X_test["sleep_hours"], 0)

X_train["excess_extracurricular"] = np.maximum(X_train["extracurricular_hours"] - 10, 0)
X_test["excess_extracurricular"] = np.maximum(X_test["extracurricular_hours"] - 10, 0)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Previous GPA** is the strongest predictor (academic momentum).
- **Study hours** has a clear positive effect (~1.2 points/hour).
- **Attendance** contributes incrementally (~0.15 per percentage point).
- **Tutoring** (Regular: +6 points vs None).
- **Parent education** has a measurable but modest effect.
- **Excessive extracurriculars** (>10 hrs) slightly reduce scores.

**Business takeaway:** Focus interventions on attendance and study hours for the biggest score improvements. Tutoring has a strong ROI.

## 26 · Limitations

1. Synthetic data with simplified academic dynamics.
2. No subject-specific difficulty.
3. Missing motivation and mental health factors.
4. No peer group effects.
5. Simplified parent education impact.

## 27 · How to Improve This Project

1. Use real anonymized educational data.
2. Add subject difficulty and curriculum features.
3. Include peer group and social network effects.
4. Model temporal study patterns (cramming vs. steady).
5. Add teacher quality and experience features.

## 28 · Production Considerations

- Deploy for early warning systems in schools.
- Trigger tutoring recommendations for at-risk students.
- Optimize class sizes based on predicted impact.
- Monitor for bias across demographic groups.
- Output prediction intervals for guidance counselors.

## 29 · Common Mistakes

1. Assuming study hours have unlimited returns (diminishing marginal effect).
2. Ignoring previous GPA (the strongest single predictor).
3. Not accounting for sleep quality.
4. Treating all extracurriculars the same (type matters).
5. Not checking for demographic fairness in predictions.

## 30 · Mini Challenge / Exercises

1. Remove `previous_gpa` — how much does R² drop?
2. Plot score vs. study_hours — is the relationship linear or logarithmic?
3. Create `study_x_tutoring` interaction feature.
4. Build separate models by school_type.
5. Check if the model is fair across `parent_education` groups.

## 31 · Final Summary / Key Takeaways

1. **Previous GPA** is the strongest score predictor.
2. **Study hours and attendance** are the most actionable levers.
3. **Tutoring** provides a measurable +6 point boost.
4. **Sleep** matters — deficit hurts performance.
5. **Excessive extracurriculars** (>10 hrs) can reduce scores.
6. **Early warning systems** based on these features can improve outcomes.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))